### Configure Visualization Inputs

What this step does: configure visualization inputs.

Required inputs: the variables and files named in the following code cell.

Produced outputs: the checked postconditions in the following code cell.

When this step may be skipped: only when those produced outputs already exist and match the requested repository revision and data split scope.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/enes/text-to-sign-production.git"
REPO_REF = "main"
RENDER_MODE = "skeleton_only"  # "skeleton_only" or "side_by_side"
CONFIDENCE_THRESHOLD = 0.05
OUTPUT_FILENAME = "visual_debug.mp4"

WORKTREE_ROOT = Path("/content/text-to-sign-production")
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/text-to-sign-production")
print("Worktree root:", WORKTREE_ROOT)
print("Drive project root:", DRIVE_PROJECT_ROOT)
print("Render mode:", RENDER_MODE)


### Mount Google Drive

What this step does: mount google drive.

Required inputs: the variables and files named in the following code cell.

Produced outputs: the checked postconditions in the following code cell.

When this step may be skipped: only when those produced outputs already exist and match the requested repository revision and data split scope.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
if not DRIVE_PROJECT_ROOT.parent.is_dir():
    raise FileNotFoundError(f"Drive MyDrive root is missing: {DRIVE_PROJECT_ROOT.parent}")
print("Drive mounted:", DRIVE_PROJECT_ROOT.parent)


### Acquire Repository

What this step does: acquire repository.

Required inputs: the variables and files named in the following code cell.

Produced outputs: the checked postconditions in the following code cell.

When this step may be skipped: only when those produced outputs already exist and match the requested repository revision and data split scope.


In [ ]:
%cd /content
_remove_output = !rm -rf "{WORKTREE_ROOT}"
_exit_code = getattr(_remove_output, "exit_code", 0)
if _exit_code != 0:
    raise RuntimeError(f"Failed to remove existing worktree: {WORKTREE_ROOT}")
if WORKTREE_ROOT.exists():
    raise RuntimeError(f"Worktree still exists after cleanup: {WORKTREE_ROOT}")

_clone_output = !git clone "{REPO_URL}" "{WORKTREE_ROOT}"
_exit_code = getattr(_clone_output, "exit_code", 0)
if _exit_code != 0:
    raise RuntimeError(f"Failed to clone repository from {REPO_URL} into {WORKTREE_ROOT}")
if not WORKTREE_ROOT.exists():
    raise RuntimeError(f"Repository clone did not create worktree: {WORKTREE_ROOT}")
if not (WORKTREE_ROOT / "pyproject.toml").is_file():
    raise RuntimeError(f"Cloned worktree is missing pyproject.toml: {WORKTREE_ROOT}")
if not (WORKTREE_ROOT / "src").is_dir():
    raise RuntimeError(f"Cloned worktree is missing src directory: {WORKTREE_ROOT}")

%cd {WORKTREE_ROOT}
_checkout_output = !git checkout "{REPO_REF}"
_exit_code = getattr(_checkout_output, "exit_code", 0)
if _exit_code != 0:
    raise RuntimeError(f"Failed to checkout revision {REPO_REF} in {WORKTREE_ROOT}")

_revision_output = !git -C "{WORKTREE_ROOT}" rev-parse HEAD
_exit_code = getattr(_revision_output, "exit_code", 0)
if _exit_code != 0:
    raise RuntimeError(f"Failed to verify checked-out revision in {WORKTREE_ROOT}")
if not (WORKTREE_ROOT / "pyproject.toml").is_file():
    raise RuntimeError(f"Checked-out worktree is missing pyproject.toml: {WORKTREE_ROOT}")
if not (WORKTREE_ROOT / "src").is_dir():
    raise RuntimeError(f"Checked-out worktree is missing src directory: {WORKTREE_ROOT}")
print("Repository ready:", WORKTREE_ROOT)


### Install And Import Package

What this step does: install and import package.

Required inputs: the variables and files named in the following code cell.

Produced outputs: the checked postconditions in the following code cell.

When this step may be skipped: only when those produced outputs already exist and match the requested repository revision and data split scope.


In [ ]:
%cd {WORKTREE_ROOT}
if not (WORKTREE_ROOT / "requirements-colab.txt").is_file():
    raise FileNotFoundError(f"Missing Colab requirements file: {WORKTREE_ROOT / 'requirements-colab.txt'}")
%pip install -r requirements-colab.txt

import sys
SRC_ROOT = WORKTREE_ROOT / "src"
if not SRC_ROOT.is_dir():
    raise RuntimeError(f"Repository post-check failed; missing src directory: {SRC_ROOT}")
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import text_to_sign_production as notebook_workflow
from text_to_sign_production.core.paths import ProjectLayout
from text_to_sign_production.core.files import (
    ensure_dir,
    require_dir,
    require_dir_contains,
    require_file,
    require_non_empty_file,
    verify_output_dir,
    verify_output_file,
)

runtime_layout = ProjectLayout(WORKTREE_ROOT)
drive_layout = ProjectLayout(DRIVE_PROJECT_ROOT)
if notebook_workflow.__name__ != "text_to_sign_production":
    raise RuntimeError("Imported unexpected package module")
if runtime_layout.data_root != WORKTREE_ROOT / "data":
    raise RuntimeError(f"Unexpected runtime data root: {runtime_layout.data_root}")
print("Package ready:", notebook_workflow.__name__)
print("Runtime data root:", runtime_layout.data_root)
print("Drive data root:", drive_layout.data_root)


### Restore Processed Manifests

What this step does: restore processed manifests.

Required inputs: the variables and files named in the following code cell.

Produced outputs: the checked postconditions in the following code cell.

When this step may be skipped: only when those produced outputs already exist and match the requested repository revision and data split scope.


In [ ]:
MANIFEST_ARCHIVE = require_non_empty_file(
    drive_layout.processed_manifests_reports_archive(),
    label="Drive processed manifest/report archive",
)
%cd {WORKTREE_ROOT}
_extract_output = !tar --zstd -xf "{MANIFEST_ARCHIVE}" -C "{WORKTREE_ROOT}"
_exit_code = getattr(_extract_output, "exit_code", 0)
if _exit_code != 0:
    raise RuntimeError(f"Failed to restore processed manifests: {MANIFEST_ARCHIVE}")
PROCESSED_MANIFEST_ROOT = require_dir_contains(
    runtime_layout.processed_v1_manifests_root,
    "*.jsonl",
    label="Runtime processed manifests",
)
for manifest_path in sorted(PROCESSED_MANIFEST_ROOT.glob("*.jsonl")):
    verify_output_file(manifest_path, label=f"Processed manifest {manifest_path.name}")
print("Processed manifests restored:", sorted(PROCESSED_MANIFEST_ROOT.glob("*.jsonl")))


### Select Manifest Sample

What this step does: select manifest sample.

Required inputs: the variables and files named in the following code cell.

Produced outputs: the checked postconditions in the following code cell.

When this step may be skipped: only when those produced outputs already exist and match the requested repository revision and data split scope.


In [ ]:
from text_to_sign_production.workflows.visualization import (
    VisualizationSelectionConfig,
    VisualizationWorkflowConfig,
    run_visualization_workflow,
    select_visualization_sample,
    validate_visualization_inputs,
)
from text_to_sign_production.visualization.skeleton import SkeletonRenderConfig
from text_to_sign_production.core.files import OutputExistsPolicy

sample_id_text = input("Sample id (leave blank for deterministic random): ").strip()
selection_config = VisualizationSelectionConfig(
    project_root=WORKTREE_ROOT,
    sample_id=sample_id_text or None,
    random_selection=not bool(sample_id_text),
    seed=13,
    require_sample_files=False,
)
selected_sample = select_visualization_sample(selection_config)
selected_split = selected_sample.record.split
print("Selected sample:", selected_sample.record.sample_id)
print("Selected split:", selected_split)
print("Selected sample path:", selected_sample.sample_path)


### Restore Selected Sample

What this step does: restore selected sample.

Required inputs: the variables and files named in the following code cell.

Produced outputs: the checked postconditions in the following code cell.

When this step may be skipped: only when those produced outputs already exist and match the requested repository revision and data split scope.


In [ ]:
SAMPLE_ARCHIVE = require_non_empty_file(
    drive_layout.processed_sample_archive(selected_split),
    label=f"Drive {selected_split} processed sample archive",
)
%cd {WORKTREE_ROOT}
_extract_output = !tar --zstd -xf "{SAMPLE_ARCHIVE}" -C "{WORKTREE_ROOT}"
_exit_code = getattr(_extract_output, "exit_code", 0)
if _exit_code != 0:
    raise RuntimeError(f"Failed to restore selected sample archive: {SAMPLE_ARCHIVE}")
selected_sample_path = verify_output_file(selected_sample.sample_path, label="Selected processed sample")
if selected_sample_path.stat().st_size <= 0:
    raise RuntimeError(f"Selected sample file is empty: {selected_sample_path}")
print("Selected sample restored:", selected_sample_path)


### Restore Source Video When Needed

What this step does: restore source video when needed.

Required inputs: the variables and files named in the following code cell.

Produced outputs: the checked postconditions in the following code cell.

When this step may be skipped: only when those produced outputs already exist and match the requested repository revision and data split scope.


In [ ]:
if RENDER_MODE == "side_by_side":
    RAW_BFH_ARCHIVE = require_non_empty_file(
        drive_layout.raw_bfh_keypoints_archive(selected_split),
        label=f"Drive {selected_split} raw BFH archive",
    )
    %cd {WORKTREE_ROOT}
    _extract_output = !tar --zstd -xf "{RAW_BFH_ARCHIVE}" -C "{WORKTREE_ROOT}"
    _exit_code = getattr(_extract_output, "exit_code", 0)
    if _exit_code != 0:
        raise RuntimeError(f"Failed to restore raw BFH archive: {RAW_BFH_ARCHIVE}")
    if selected_sample.source_video_path is None:
        raise FileNotFoundError("Selected sample has no source video path")
    verify_output_file(selected_sample.source_video_path, label="Selected source video")
    print("Source video restored:", selected_sample.source_video_path)
else:
    print("Skeleton-only render selected; raw BFH video restore skipped.")


### Run Visualization Workflow

What this step does: run visualization workflow.

Required inputs: the variables and files named in the following code cell.

Produced outputs: the checked postconditions in the following code cell.

When this step may be skipped: only when those produced outputs already exist and match the requested repository revision and data split scope.


In [ ]:
render_selection_config = VisualizationSelectionConfig(
    project_root=WORKTREE_ROOT,
    splits=(selected_split,),
    sample_id=selected_sample.record.sample_id,
    require_sample_files=True,
)
workflow_config = VisualizationWorkflowConfig(
    selection=render_selection_config,
    mode=RENDER_MODE,
    output_filename=OUTPUT_FILENAME,
    skeleton_config=SkeletonRenderConfig(confidence_threshold=CONFIDENCE_THRESHOLD),
    output_policy=OutputExistsPolicy.OVERWRITE,
)
validate_visualization_inputs(workflow_config)
visualization_result = run_visualization_workflow(workflow_config)
verify_output_file(visualization_result.output_path, label="Runtime visualization MP4")
if not visualization_result.output_path.is_relative_to(runtime_layout.visualization_artifacts_root):
    raise RuntimeError(f"Visualization output escaped artifact root: {visualization_result.output_path}")
print("Visualization output:", visualization_result.output_path)


### Publish Visualization Output

What this step does: publish visualization output.

Required inputs: the variables and files named in the following code cell.

Produced outputs: the checked postconditions in the following code cell.

When this step may be skipped: only when those produced outputs already exist and match the requested repository revision and data split scope.


In [ ]:
VISUALIZATION_OUTPUT_DRIVE_ROOT = drive_layout.visualization_artifacts_root
_mkdir_output = !mkdir -p "{VISUALIZATION_OUTPUT_DRIVE_ROOT}"
_exit_code = getattr(_mkdir_output, "exit_code", 0)
if _exit_code != 0:
    raise RuntimeError(f"Failed to create visualization Drive output directory: {VISUALIZATION_OUTPUT_DRIVE_ROOT}")
require_dir(VISUALIZATION_OUTPUT_DRIVE_ROOT, label="Visualization Drive output directory")
PUBLISHED_VISUALIZATION_PATH = VISUALIZATION_OUTPUT_DRIVE_ROOT / visualization_result.output_path.name
_copy_output = !cp "{visualization_result.output_path}" "{VISUALIZATION_OUTPUT_DRIVE_ROOT}/"
_exit_code = getattr(_copy_output, "exit_code", 0)
if _exit_code != 0:
    raise RuntimeError(f"Failed to publish visualization output to Drive: {VISUALIZATION_OUTPUT_DRIVE_ROOT}")
verify_output_file(PUBLISHED_VISUALIZATION_PATH, label="Published visualization MP4")
if PUBLISHED_VISUALIZATION_PATH.stat().st_size <= 0:
    raise RuntimeError(f"Published visualization MP4 is empty: {PUBLISHED_VISUALIZATION_PATH}")
print("Published visualization:", PUBLISHED_VISUALIZATION_PATH)
